In [1]:
import pandas as pd
import re
import numpy as np

In [2]:
df = pd.read_csv("data/herm_full_edgelist_MODIFIED.csv")
print(f"Dataset dimensions: {df.shape}")
df.head()

Dataset dimensions: (7394, 4)


,Source,Target,Weight,Type
0,I1L,I2L,10,chemical
1,I1L,I3,3,chemical
2,I1L,I5,2,chemical
3,I1L,I6,1,chemical
4,I1L,M2L,3,chemical


Check for missing values and duplicates

In [3]:
duplicates = df.duplicated().sum()
missings = df.isnull().sum()

print(f"Duplicates: {duplicates}")
print(f"Missing values: \n{missings}")

Duplicates: 0
Missing values: 
Source    0
Target    0
Weight    0
Type      0
dtype: int64


Dataset has no missing values or duplicates. Next, we clean the data so that it is easier to work with.

In [ ]:
# Encode the types column into binary
print(df["Type"].unique())
print(df["Type"].head())
df["IsElectrical"] = df["Type"].astype("category").cat.codes
print(df["IsElectrical"].head())
df["IsElectrical"].unique()
df.head()

['chemical' 'electrical']
0    chemical
1    chemical
2    chemical
3    chemical
4    chemical
Name: Type, dtype: object
0    0
1    0
2    0
3    0
4    0
Name: IsElectrical, dtype: int8


,Source,Target,Weight,Type,IsElectrical
0,I1L,I2L,10,chemical,0
1,I1L,I3,3,chemical,0
2,I1L,I5,2,chemical,0
3,I1L,I6,1,chemical,0
4,I1L,M2L,3,chemical,0


In [5]:
# Encode Left vs. Right
df["Source"] = df["Source"].str.strip()
df["Target"] = df["Target"].str.strip()
not_LR = ["PQR", "AVL", "PVR", "RIR"] # https://www.wormatlas.org/neurons/Individual%20Neurons/Neuronframeset.html

for i in range(df.shape[0]):
    if re.match(r"^.*L$",df.loc[i, "Source"]) and (df.loc[i, "Source"] not in not_LR):
        df.loc[i, "isSourceLeft"] = True
    else:
        df.loc[i, "isSourceLeft"] = False

    if re.match(r"^.*R$",df.loc[i, "Source"]) and (df.loc[i, "Source"] not in not_LR):
        df.loc[i, "isSourceRight"] = True
    else:
        df.loc[i, "isSourceRight"] = False
        
    if re.match(r"^.*L$",df.loc[i, "Target"]) and (df.loc[i, "Target"] not in not_LR):
        df.loc[i, "isTargetLeft"] = True
    else:
        df.loc[i, "isTargetLeft"] = False

    if re.match(r"^.*R$",df.loc[i, "Target"]) and (df.loc[i, "Target"] not in not_LR):
        df.loc[i, "isTargetRight"] = True
    else:
        df.loc[i, "isTargetRight"] = False

df.head(40)


,Source,Target,Weight,Type,IsElectrical,isSourceLeft,isSourceRight,isTargetLeft,isTargetRight
0,I1L,I2L,10,chemical,0,True,False,True,False
1,I1L,I3,3,chemical,0,True,False,False,False
2,I1L,I5,2,chemical,0,True,False,False,False
3,I1L,I6,1,chemical,0,True,False,False,False
4,I1L,M2L,3,chemical,0,True,False,True,False
5,I1L,M3L,8,chemical,0,True,False,True,False
6,I1L,M3R,2,chemical,0,True,False,False,True
7,I1L,MCL,2,chemical,0,True,False,True,False
8,I1L,MCR,2,chemical,0,True,False,False,True
9,I1L,MI,2,chemical,0,True,False,False,False


In [ ]:
# Encode Neuron Types